# NeurALPS · compatibility and activity

**Two complementary outputs from the frozen SSL26 encoder.**

- **Pretrained compatibility:** local and full-context raw reconstruction maps. Select the exchanged domains to obtain the newer joint-mask region scores.
- **Supervised activity:** the typed-context logistic head fitted on **494 reviewed engineered assemblies** — BODE1 324, BODE2 105, T-domain 65.

The combined head is fitted and included. New-sequence inference needs the runtime asset ZIP exported from Jean Zay. Its independent golden checks run below. Exact ESMC extraction requires a BF16-capable NVIDIA GPU (for example A100 or L4); the saved-feature demonstration and encoder checks can run on CPU.

Input: complete proteins in biosynthetic order, with approved domain types and **0-based, end-exclusive** coordinates. Domain calling and biosynthetic ordering are not inferred by this notebook.


## 1 · Load the release
Upload the release ZIP, or use a repository URL and a pinned 40-character commit after publication.

In [ ]:
SOURCE = "upload" #@param ["upload", "github"]
REPO_URL = "" #@param {type:"string"}
COMMIT = "" #@param {type:"string"}
from pathlib import Path
import subprocess, sys, json, hashlib, zipfile, stat, re
BASE = Path("/content/neuralps_work")
BASE.mkdir(exist_ok=True)
def safe_extract(archive, destination):
    if destination.exists():
        raise FileExistsError(f"Use a fresh destination: {destination}")
    with zipfile.ZipFile(archive) as z:
        if sum(i.file_size for i in z.infolist()) > 512*1024**2:
            raise ValueError("Archive exceeds the runtime bundle size limit")
        for i in z.infolist():
            p = Path(i.filename)
            if p.is_absolute() or ".." in p.parts or stat.S_ISLNK(i.external_attr >> 16):
                raise ValueError("Unsafe ZIP member")
        z.extractall(destination)
if SOURCE == "upload":
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload one NeurALPS_Colab_Release ZIP")
    archive = BASE / "release.zip"
    archive.write_bytes(next(iter(uploaded.values())))
    safe_extract(archive, BASE/"release")
    candidates = list((BASE/"release").glob("*/pyproject.toml"))
    if len(candidates) != 1:
        raise ValueError("Release root was not found uniquely")
    ROOT = candidates[0].parent
else:
    if not REPO_URL.startswith("https://github.com/") or not re.fullmatch("[0-9a-f]{40}", COMMIT):
        raise ValueError("Provide a GitHub URL and exact commit")
    ROOT = BASE/"repository"
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(ROOT)], check=True)
    subprocess.run(["git", "checkout", "--detach", COMMIT], cwd=ROOT, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT), "matplotlib", "huggingface-hub>=0.34,<1"], check=True)
sys.path.insert(0, str(ROOT))
print("Release:", ROOT)


## 2 · Inspect the fitted supervised model
This demonstration uses saved training features. These predictions are **in sample**, and do not measure generalization.

In [ ]:
import numpy as np
from scipy.special import expit
from neuralps.head import load_model, predict
from neuralps.contracts import sha
HEAD = ROOT/"models/all494"
head_manifest = json.loads((HEAD/"manifest.json").read_text())
assert sha(HEAD/"supervised_head.npz") == head_manifest["model_sha256"]
print("Training rows:", head_manifest["trained_rows"], head_manifest["by_dataset"])
print("L2 penalty:", head_manifest["l2"], "— encoder updates:", head_manifest["encoder_updates"])
rows = json.loads((ROOT/"data/training_rows.json").read_text())
with np.load(ROOT/"data/training_features.npz", allow_pickle=False) as z:
    TRAIN_X = z["features"]
head = load_model(HEAD/"supervised_head.npz")
example_index = 0 #@param {type:"integer"}
if not 0 <= example_index < len(rows):
    raise ValueError("example_index must be 0–493")
print(rows[example_index])
print("In-sample supervised activity score:", float(expit(predict(head, TRAIN_X[example_index:example_index+1])[0])))


## 3 · Load runtime assets and verify encoder parity
Upload **NeurALPS_runtime_20261009.zip**, created by `scripts/export_runtime.py`. It contains the frozen encoder, training-only centering means, historical embedding references, and the installed Transformers fork. It contains no ESMC weights or training caches.

In [ ]:
from google.colab import files
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload one runtime asset ZIP")
archive = BASE/"runtime.zip"
archive.write_bytes(next(iter(uploaded.values())))
ASSETS = BASE/"runtime_assets"
safe_extract(archive, ASSETS)
manifest = json.loads((ASSETS/"manifest.json").read_text())
for name, digest in manifest["files_sha256"].items():
    target = (ASSETS/name).resolve()
    if not target.is_relative_to(ASSETS.resolve()) or sha(target) != digest:
        raise ValueError("Runtime checksum mismatch: "+name)
subprocess.run([sys.executable, str(ROOT/"scripts/verify_runtime.py"), "--assets", str(ASSETS), "--report", str(BASE/"encoder_parity.json")], check=True)


## 4 · Install the preserved ESMC implementation
The installed fork is exported with its metadata and licenses because its recorded Git URL was unavailable during porting. Every Python source file is checked before inference.

In [ ]:
if "transformers" in sys.modules:
    raise RuntimeError("Restart the runtime before installing the preserved Transformers fork, then rerun setup")
wheel = ASSETS/manifest["transformers_distribution"]["wheel"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tokenizers==0.22.1", "safetensors>=0.4.3", "filelock", "packaging", "pyyaml", "regex", "requests", "tqdm", "huggingface-hub>=0.34,<1"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps", str(wheel)], check=True)
import torch
if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
    raise RuntimeError("Select a BF16-capable GPU runtime for exact ESMC extraction. The encoder checks above work on CPU.")
print(torch.cuda.get_device_name(0))


## 5 · Obtain the pinned ESMC weights
The download uses the original model revision and verifies the weight SHA-256. If you already have the same model folder on mounted storage, enter its path.

In [ ]:
LOCAL_ESMC_DIRECTORY = "" #@param {type:"string"}
from neuralps.embedding import download_esmc
ESMC = Path(LOCAL_ESMC_DIRECTORY) if LOCAL_ESMC_DIRECTORY else download_esmc(BASE/"ESMC-600M")
print("Pinned model folder:", ESMC)


## 6 · Choose an annotated assembly
The included AI_1 example reproduces the original physical input and is part of the supervised training cohort. For your own assembly, upload an annotated JSON following the example. `joint_domain_indices` refers to domain objects numbered 0, 2, 4, …; it must cover a contiguous exchanged region. Touching boundaries are included automatically.

In [ ]:
INPUT = "example" #@param ["example", "upload JSON"]
if INPUT == "example":
    spec = json.loads((ROOT/"examples/AI_1_annotated.json").read_text())
else:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one assembly JSON")
    spec = json.loads(next(iter(uploaded.values())).decode("utf-8"))
from neuralps.inputs import build_record
record, parents, sequences = build_record(spec)
print(spec["assembly_id"], "—", record["n_chains"], "physical chains;", record["n_domains"], "domains")
for j, obj in enumerate(record["route"]):
    if obj["kind"] == 0:
        print(j, obj["canonical_domain_type"], obj["protein_uid"], obj["start_aa_0based"], obj["end_aa_0based_exclusive"])
print("Joint region domain indices:", spec.get("joint_domain_indices", "not selected"))


## 7 · Run the two outputs
The pretrained map masks each physical object and its overlapping/identical sequence features. A selected joint region hides the entire region and touching boundaries together. The supervised head always uses the clean full-context representation, exactly as in its training feature contract.

In [ ]:
OUTPUTS = "both" #@param ["both", "pretrained only"]
from neuralps.cli import analyze
result = analyze(spec, ASSETS, HEAD if OUTPUTS == "both" else None, ESMC, device="cpu")
from neuralps.plotting import plot_result
fig = plot_result(result)
from IPython.display import display
display(fig)
if "joint_region" in result:
    print("Joint-mask status:", result["joint_region"]["status"])
    print(json.dumps(result["joint_region"]["aggregate"], indent=2))
if "supervised" in result:
    print("Supervised activity score:", result["supervised"]["activity_score"])


**How to read the outputs**

- Raw reconstruction agreement is between −1 and 1. It is not an activity probability. Unsupported objects stay unscored with a reason.
- The promising BODE2 result used a balanced joint-region score; the T-domain neighborhood result used a different aggregation. The single-object map is not assigned those AUROCs.
- Supervised activity is an uncalibrated sigmoid score. Signed local logit contributions explain the classifier; they do not identify a causal defective domain.
- No natural-reference percentiles are assigned to an unsupported scoring/masking contract.


## 8 · Download the result

In [ ]:
from neuralps.contracts import dump
result_path = BASE/"neuralps_result.json"
dump(result_path, result)
figure_path = BASE/"neuralps_map.png"
fig.savefig(figure_path, dpi=180, bbox_inches="tight")
files.download(str(result_path))
files.download(str(figure_path))


## Optional · Reproduce the all-label fit
The saved head is already trained. This refits the same 494 rows and checks exact membership before optimization. The output directory must be new. It does not create a new held-out evaluation.

In [ ]:
REFIT = False #@param {type:"boolean"}
if REFIT:
    subprocess.run([sys.executable, str(ROOT/"scripts/fit_all_labels.py"), "--output", str(BASE/"refit_all494")], check=True)
